In [16]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
import torch

In [17]:
# ==============================
# 1. Load Base Encoders (without classification heads)
# ==============================
from transformers import RobertaModel, AutoModel
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CodeBERT encoder (semantic)
codebert_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")
codebert_encoder.to(device)
codebert_encoder.eval()  # we will train the whole model, but keep encoder trainable

# UniXcoder encoder (syntactic)
unixcoder_encoder = AutoModel.from_pretrained("microsoft/unixcoder-base")
unixcoder_encoder.to(device)
unixcoder_encoder.eval()


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(51416, 768, padding_idx=1)
    (position_embeddings): Embedding(1026, 768, padding_idx=1)
    (token_type_embeddings): Embedding(10, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (

In [18]:
# ==============================
# 2. Pooling function (masked average pooling)
# ==============================
def masked_average_pooling(hidden_states, attention_mask):
    """
    hidden_states: (batch_size, seq_len, hidden_dim)
    attention_mask: (batch_size, seq_len) with 1 for real tokens, 0 for padding
    Returns: (batch_size, hidden_dim)
    """
    # Expand mask to hidden_dim dimension
    mask = attention_mask.unsqueeze(-1).float()  # (B, L, 1)
    sum_hidden = torch.sum(hidden_states * mask, dim=1)   # (B, H)
    sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)     # (B, 1)
    return sum_hidden / sum_mask


In [19]:


# ==============================
# 3. Adapter layers (for aligning feature spaces)
# ==============================
class Adapter(nn.Module):
    def __init__(self, input_dim, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = input_dim
        self.linear = nn.Linear(input_dim, hidden_dim)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        x = self.linear(x)
        x = self.act(x)
        x = self.norm(x)
        return x

# ==============================
# 4. Fusion layer (concatenate + linear + activation + dropout + norm)
# ==============================
class FusionLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(output_dim)

    def forward(self, x):
        x = self.linear(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.norm(x)
        return x

# ==============================
# 5. Two-layer classifier head
# ==============================
class ClassifierHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_labels=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, num_labels)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        logits = self.fc2(x)
        return logits

# ==============================
# 6. Complete Dual Encoder Model
# ==============================
class DualEncoderModel(nn.Module):
    def __init__(self, codebert_encoder, unixcoder_encoder, hidden_dim=768, fusion_dim=768, num_labels=2):
        super().__init__()
        self.codebert = codebert_encoder
        self.unixcoder = unixcoder_encoder

        # Adapters (input dim = hidden_dim from both encoders)
        self.codebert_adapter = Adapter(hidden_dim, hidden_dim)
        self.unixcoder_adapter = Adapter(hidden_dim, hidden_dim)

        # Fusion: input dim = 2 * hidden_dim, output dim = fusion_dim
        self.fusion = FusionLayer(2 * hidden_dim, fusion_dim)

        # Classifier: input fusion_dim -> hidden_dim (e.g., 768 -> 384) -> num_labels
        self.classifier = ClassifierHead(fusion_dim, fusion_dim // 2, num_labels)

    def forward(self, codebert_input_ids, codebert_attention_mask,
                unixcoder_input_ids, unixcoder_attention_mask):
        # 1. CodeBERT forward
        codebert_outputs = self.codebert(
            input_ids=codebert_input_ids,
            attention_mask=codebert_attention_mask
        )
        codebert_hidden = codebert_outputs.last_hidden_state  # (B, L, H)
        S = masked_average_pooling(codebert_hidden, codebert_attention_mask)  # (B, H)
        S = self.codebert_adapter(S)  # (B, H)

        # 2. UniXcoder forward
        unixcoder_outputs = self.unixcoder(
            input_ids=unixcoder_input_ids,
            attention_mask=unixcoder_attention_mask
        )
        unixcoder_hidden = unixcoder_outputs.last_hidden_state  # (B, L, H)
        A = masked_average_pooling(unixcoder_hidden, unixcoder_attention_mask)  # (B, H)
        A = self.unixcoder_adapter(A)  # (B, H)

        # 3. Fusion: concatenate and transform
        Z = torch.cat([S, A], dim=-1)  # (B, 2H)
        F = self.fusion(Z)             # (B, fusion_dim)

        # 4. Classification
        logits = self.classifier(F)    # (B, num_labels)
        return logits


In [20]:
# ==============================
# 7. Dataset that returns tokenizations for both models
# ==============================
class DualCodeDataset(torch.utils.data.Dataset):
    def __init__(self, codes, labels, tokenizer_cb, tokenizer_uc, max_length=512):
        self.codes = codes
        self.labels = labels
        self.tokenizer_cb = tokenizer_cb
        self.tokenizer_uc = tokenizer_uc
        self.max_length = max_length

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        code = self.codes[idx]
        label = self.labels[idx]

        # CodeBERT tokenization
        enc_cb = self.tokenizer_cb(
            code,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        # UniXcoder tokenization
        enc_uc = self.tokenizer_uc(
            code,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {
            'codebert_input_ids': enc_cb['input_ids'].squeeze(0),
            'codebert_attention_mask': enc_cb['attention_mask'].squeeze(0),
            'unixcoder_input_ids': enc_uc['input_ids'].squeeze(0),
            'unixcoder_attention_mask': enc_uc['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }
        return item

In [21]:
tokenizer_cb = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
tokenizer_uc = AutoTokenizer.from_pretrained("microsoft/unixcoder-base")

In [22]:
# ==============================
# 10. Modified DualEncoderModel that returns fused features
# ==============================
class DualEncoderModelWithFeatures(DualEncoderModel):
    def forward(self, codebert_input_ids, codebert_attention_mask,
                unixcoder_input_ids, unixcoder_attention_mask,
                return_features=False):
        # CodeBERT forward
        codebert_outputs = self.codebert(
            input_ids=codebert_input_ids,
            attention_mask=codebert_attention_mask
        )
        codebert_hidden = codebert_outputs.last_hidden_state
        S = masked_average_pooling(codebert_hidden, codebert_attention_mask)
        S = self.codebert_adapter(S)

        # UniXcoder forward
        unixcoder_outputs = self.unixcoder(
            input_ids=unixcoder_input_ids,
            attention_mask=unixcoder_attention_mask
        )
        unixcoder_hidden = unixcoder_outputs.last_hidden_state
        A = masked_average_pooling(unixcoder_hidden, unixcoder_attention_mask)
        A = self.unixcoder_adapter(A)

        # Fusion
        Z = torch.cat([S, A], dim=-1)
        F = self.fusion(Z)

        # Classification
        logits = self.classifier(F)

        if return_features:
            return logits, F
        return logits

In [23]:
# ==============================
# 11. Loss functions
# ==============================
def kl_divergence(p, q):
    """Symmetric KL divergence between two probability distributions."""
    p = p.clamp(min=1e-8, max=1-1e-8)  # avoid log(0)
    q = q.clamp(min=1e-8, max=1-1e-8)
    return 0.5 * (torch.sum(p * (p.log() - q.log()), dim=-1) +
                  torch.sum(q * (q.log() - p.log()), dim=-1)).mean()

def contrastive_loss(features, labels):
    """
    Contrastive loss for different classes: L_CL = max(0, cos(u, v))
    where u, v are features of samples with different labels.
    """
    # Normalize features for cosine similarity
    features = nn.functional.normalize(features, dim=-1)
    batch_size = features.size(0)
    # Compute pairwise cosine similarity matrix
    sim_matrix = torch.mm(features, features.t())  # (B, B)
    # Create mask for different labels
    labels = labels.view(-1, 1)
    diff_mask = (labels != labels.t()).float()  # 1 if different, 0 if same
    # Only consider pairs with different labels (upper triangular to avoid double count)
    triu_mask = torch.triu(torch.ones_like(diff_mask), diagonal=1).float()
    mask = diff_mask * triu_mask
    # Loss: max(0, cos) for those pairs, average over number of pairs
    loss = torch.relu(sim_matrix)  # max(0, cos)
    loss = (loss * mask).sum() / (mask.sum() + 1e-8)
    return loss


In [24]:
# ==============================
# 12. Evaluation metrics
# ==============================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_metrics(preds, labels, probs=None):
    metrics = {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
    }
    if probs is not None:
        metrics['auroc'] = roc_auc_score(labels, probs[:, 1])
    return metrics


In [25]:

def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids_cb = batch['codebert_input_ids'].to(device)
            mask_cb = batch['codebert_attention_mask'].to(device)
            input_ids_uc = batch['unixcoder_input_ids'].to(device)
            mask_uc = batch['unixcoder_attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids_cb, mask_cb, input_ids_uc, mask_uc)
            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(probs, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    metrics = compute_metrics(all_preds, all_labels, np.array(all_probs))
    return metrics


In [26]:
# ==============================
# 13. Training and validation loops
# ==============================
def train_epoch(model, dataloader, optimizer, device, alpha=0.1, beta=0.2):
    model.train()
    total_loss = 0
    total_ce = 0
    total_kl = 0
    total_cl = 0

    for batch in dataloader:
        optimizer.zero_grad()

        # Move to device
        input_ids_cb = batch['codebert_input_ids'].to(device)
        mask_cb = batch['codebert_attention_mask'].to(device)
        input_ids_uc = batch['unixcoder_input_ids'].to(device)
        mask_uc = batch['unixcoder_attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # --- Cross-entropy loss ---
        logits = model(input_ids_cb, mask_cb, input_ids_uc, mask_uc)
        ce_loss = nn.CrossEntropyLoss()(logits, labels)

        # --- Consistency loss (KL) ---
        # Two forward passes with dropout (model.train() already active)
        logits1, features1 = model(input_ids_cb, mask_cb, input_ids_uc, mask_uc, return_features=True)
        logits2, features2 = model(input_ids_cb, mask_cb, input_ids_uc, mask_uc, return_features=True)

        probs1 = torch.softmax(logits1, dim=-1)
        probs2 = torch.softmax(logits2, dim=-1)
        kl_loss = kl_divergence(probs1, probs2)

        # --- Contrastive loss ---
        # Use features from the first forward pass
        cl_loss = contrastive_loss(features1, labels)

        # Total loss
        loss = ce_loss + alpha * kl_loss + beta * cl_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_ce += ce_loss.item()
        total_kl += kl_loss.item()
        total_cl += cl_loss.item()

    return total_loss / len(dataloader), total_ce / len(dataloader), total_kl / len(dataloader), total_cl / len(dataloader)


In [27]:
# ==============================
# 13.5 Load raw data and create train/val/test splits
# ==============================
from sklearn.model_selection import train_test_split
import os

# Path to the folder containing Label_0 and Label_1 directories
# Adjust this to your actual data path
DATA_ROOT = "../Text_Files/Train"

def load_all_data(data_root):
    codes = []
    labels = []
    for label_dir in ["Label_0", "Label_1"]:
        label_path = os.path.join(data_root, label_dir)
        if not os.path.exists(label_path):
            print(f"Warning: Directory {label_path} not found. Skipping.")
            continue
        for filename in os.listdir(label_path):
            if filename.endswith(".txt"):
                filepath = os.path.join(label_path, filename)
                with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                    code = f.read()
                codes.append(code)
                labels.append(0 if label_dir == "Label_0" else 1)
    return codes, labels

# Load all data
all_codes, all_labels = load_all_data(DATA_ROOT)
print(f"Loaded {len(all_codes)} samples")

# First split: 80% train, 20% temporary (test+val)
train_codes, val_codes, train_labels, val_labels = train_test_split(
    all_codes, all_labels, test_size=0.1, random_state=42, stratify=all_labels
)

test_codes,  test_labels  = load_all_data("../Text_Files/Test_0")


print(f"Train: {len(train_codes)}, Val: {len(val_codes)}, Test: {len(test_codes)}")

Loaded 6190 samples
Train: 5571, Val: 619, Test: 1002


In [28]:
# ==============================
# Create DualCodeDataset instances
# ==============================
train_dataset = DualCodeDataset(train_codes, train_labels, tokenizer_cb, tokenizer_uc)
val_dataset   = DualCodeDataset(val_codes,   val_labels,   tokenizer_cb, tokenizer_uc)
test_dataset  = DualCodeDataset(test_codes,  test_labels,  tokenizer_cb, tokenizer_uc)

print("Datasets created.")

Datasets created.


In [29]:
# ==============================
# 14. Create DataLoaders
# ==============================
from torch.utils.data import DataLoader
import numpy as np

# Assuming train_dataset, val_dataset, test_dataset are already defined
# as instances of DualCodeDataset. If not, create them first.

# Create DataLoaders
batch_size = 8  # as per paper

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [30]:
# ==============================
# 15. Training Loop
# ==============================
# Initialize model
model = DualEncoderModelWithFeatures(codebert_encoder, unixcoder_encoder).to(device)

# Optimizer (AdamW as per paper)
optimizer = torch.optim.AdamW(model.parameters(), lr=2.5e-5, eps=1e-8)

# Training hyperparameters
num_epochs = 3
alpha = 0.1   # weight for KL loss
beta = 0.2    # weight for contrastive loss

print("Starting training...")
for epoch in range(num_epochs):
    # Train for one epoch
    train_loss, ce_loss, kl_loss, cl_loss = train_epoch(
        model, train_loader, optimizer, device, alpha, beta
    )

    # Evaluate on validation set
    val_metrics = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} (CE: {ce_loss:.4f}, KL: {kl_loss:.4f}, CL: {cl_loss:.4f})")
    print(f"  Val F1: {val_metrics['f1']:.4f}, Acc: {val_metrics['accuracy']:.4f}, AUROC: {val_metrics['auroc']:.4f}")

Starting training...
Epoch 1/3
  Train Loss: 0.3990 (CE: 0.3715, KL: 0.0112, CL: 0.1323)
  Val F1: 0.8850, Acc: 0.8821, AUROC: 0.9543
Epoch 2/3
  Train Loss: 0.2459 (CE: 0.2313, KL: 0.0232, CL: 0.0614)
  Val F1: 0.8793, Acc: 0.8821, AUROC: 0.9600
Epoch 3/3
  Train Loss: 0.1592 (CE: 0.1499, KL: 0.0233, CL: 0.0347)
  Val F1: 0.9152, Acc: 0.9144, AUROC: 0.9672


In [31]:
# Final evaluation on test set
print("\nEvaluating on test set...")
test_metrics = evaluate(model, test_loader, device)
print("Test Results:")
for key, value in test_metrics.items():
    print(f"  {key}: {value:.4f}")


Evaluating on test set...
Test Results:
  accuracy: 0.8413
  precision: 0.8685
  recall: 0.8044
  f1: 0.8352
  auroc: 0.9351
